# MOOC 01 — Données, PDE et pools

But : comprendre les tableaux `.npz` manipulés par le projet.

On part du run local `runs/smoke`, car il est petit et présent dans le dépôt. Le même code marche sur des runs Bigfoot/Leonardo plus lourds.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RUN = ROOT / "runs" / "smoke"
print(RUN)

## 1. Schéma des fichiers `.npz`

`validation.npz` contient des trajectoires complètes.

`pool_round_*.npz` contient des transitions aplaties : `states`, `params`, `next_states`, `losses`, `source`.

In [ ]:
def describe_npz(path: Path):
    print(f"\n--- {path.relative_to(ROOT)} ---")
    if not path.exists():
        print("missing")
        return
    with np.load(path, allow_pickle=True) as data:
        for key in data.files:
            arr = data[key]
            if arr.dtype == object:
                finite = "object"
            else:
                finite = f"finite={np.isfinite(arr).mean():.3f}"
            print(f"{key:18s} shape={str(arr.shape):18s} dtype={str(arr.dtype):8s} {finite}")
        if "source" in data.files:
            values, counts = np.unique(data["source"], return_counts=True)
            print("source counts:", dict(zip(values.astype(int), counts.astype(int))))

for path in [RUN / "validation.npz", RUN / "pool_round_0.npz", RUN / "pool_round_1.npz"]:
    describe_npz(path)

## 2. Validation : trajectoires Halton

La validation est fixe et déterministe. Elle sert à comparer les rounds sans changer le banc de test.

In [ ]:
val_path = RUN / "validation.npz"
with np.load(val_path) as val:
    states0 = val["states0"]
    params = val["params"]
    trajectories = val["trajectories"]

print("states0", states0.shape)
print("params", params.shape)
print("trajectories", trajectories.shape)
print("n_trajectories", trajectories.shape[0], "steps", trajectories.shape[1] - 1, "resolution", trajectories.shape[-1])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
idx = 0
axes[0].imshow(trajectories[idx, :, 0, :], aspect="auto", origin="lower")
axes[0].set_title(f"Trajectoire validation #{idx}")
axes[0].set_xlabel("x")
axes[0].set_ylabel("t")
for t in [0, 1, min(3, trajectories.shape[1]-1), trajectories.shape[1]-1]:
    axes[1].plot(trajectories[idx, t, 0], label=f"t={t}")
axes[1].set_title("Profils spatiaux")
axes[1].legend()
plt.tight_layout()

## 3. Pool round 0 : uniforme uniquement

Round 0 utilise toujours des trajectoires uniformes. Les transitions sont ensuite aplaties.

In [ ]:
def load_pool(round_id: int):
    path = RUN / f"pool_round_{round_id}.npz"
    with np.load(path, allow_pickle=True) as data:
        out = {key: data[key] for key in data.files}
    return out

pool0 = load_pool(0)
for key in ["states", "params", "next_states", "losses", "source", "pretrain_losses"]:
    arr = pool0[key]
    print(f"{key:16s}", arr.shape, arr.dtype)

n_samples = len(pool0["states"])
print("n_samples", n_samples)
print("source unique", np.unique(pool0["source"], return_counts=True))

In [ ]:
def total_variation(states):
    s = np.asarray(states).reshape(len(states), -1)
    return np.abs(np.diff(s, axis=1)).sum(axis=1) + np.abs(s[:, 0] - s[:, -1])

def rms(states):
    s = np.asarray(states).reshape(len(states), -1)
    return np.sqrt(np.mean(s ** 2, axis=1))

for name, arr in [("state", pool0["states"]), ("next", pool0["next_states"] )]:
    tv = total_variation(arr)
    amp = rms(arr)
    print(name, "TV mean", tv.mean(), "TV p90", np.quantile(tv, 0.9), "RMS mean", amp.mean())

## 4. Pool round 1 : mélange uniforme + généré

Dans le smoke : `uniform_fraction=0.5`, donc round 1 mélange 12 transitions uniformes et 12 transitions générées.

Codes source :

- `0` : trajectoire uniforme ;
- `1` : état généré ou stratégie non-uniforme ;
- `2` : fallback uniforme après échec de génération/solveur.

In [ ]:
pool1 = load_pool(1)
source = pool1["source"]
values, counts = np.unique(source, return_counts=True)
print(dict(zip(values.astype(int), counts.astype(int))))

for src_id, label in [(0, "uniform"), (1, "generated"), (2, "fallback")]:
    mask = source == src_id
    if not mask.any():
        continue
    print(f"\n{label}")
    print("  count", int(mask.sum()))
    print("  loss mean", float(np.nanmean(pool1["losses"][mask])))
    print("  pretrain loss mean", float(np.nanmean(pool1["pretrain_losses"][mask])))
    print("  TV mean", float(total_variation(pool1["states"][mask]).mean()))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.ravel()
axes[0].hist(pool1["losses"], bins=12)
axes[0].set_title("losses round 1")
axes[1].hist(pool1["pretrain_losses"], bins=12)
axes[1].set_title("pretrain losses round 1")

for src_id, label, ax in [(0, "uniform", axes[2]), (1, "generated", axes[3])]:
    mask = pool1["source"] == src_id
    if mask.any():
        ax.plot(pool1["states"][mask][0, 0], label="state")
        ax.plot(pool1["next_states"][mask][0, 0], label="next")
    ax.set_title(label)
    ax.legend()
plt.tight_layout()

## 5. Vérifier l'invariant de budget

L'idée centrale est de comparer à budget solveur comparable. On garde donc `n_samples = trajectory_steps * n_trajectories`.

In [ ]:
config = json.loads((RUN / "config.resolved.json").read_text())
T = config["pool"]["trajectory_steps"]
N = config["pool"]["n_trajectories"]
expected = T * N
print("T", T, "N", N, "expected", expected)
for round_id in [0, 1]:
    pool = load_pool(round_id)
    print(f"round {round_id}: {len(pool['states'])} samples", "OK" if len(pool["states"]) == expected else "MISMATCH")

## 6. Ce qu'il faut regarder dans un nouveau run

Pour chaque `pool_round_*.npz` :

- `source` : la stratégie remplit-elle bien la fraction attendue ?
- `losses` et `pretrain_losses` : les états ciblés sont-ils plus difficiles avant apprentissage ?
- TV/RMS/spectre : les états générés restent-ils physiquement plausibles ?
- `target_bins` si présent : les bins demandés sont-ils bien persistés ?